# 03 · Basic Data Cleaning

## Objective
Fix the data-quality issues recorded in notebook 02, justify every change, and save a clean
dataset (`data/churn_cleaned.csv`) that all later notebooks use.

Rule followed here: **investigate first, then decide.** No rows are deleted blindly.

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use src/churn_utils.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.churn_utils import CLEAN_DATA_PATH, RAW_DATA_PATH

df = pd.read_csv(RAW_DATA_PATH)
print("Raw shape:", df.shape)

Raw shape: (7043, 21)


## 1. `TotalCharges`: incorrect data type + hidden missing values

`TotalCharges` should be numeric but was read as text. Convert it with `errors="coerce"` so that
values that can't be parsed become `NaN`, then **look at those rows** before deciding anything.

In [2]:
total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
bad_rows = df[total_charges_numeric.isna()]
print("Rows that could not be converted:", len(bad_rows))
bad_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

Rows that could not be converted: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


All unparseable values are blanks, and **every one of them has `tenure = 0`**. Let's confirm
that `tenure = 0` occurs *only* in these rows, and check how `TotalCharges` relates to
`tenure × MonthlyCharges` for everyone else.

In [3]:
print("Customers with tenure == 0:", (df["tenure"] == 0).sum())

valid = df[total_charges_numeric.notna()]
approx_total = valid["tenure"] * valid["MonthlyCharges"]
print("Correlation(TotalCharges, tenure x MonthlyCharges):",
      round(total_charges_numeric[valid.index].corr(approx_total), 4))

Customers with tenure == 0: 11
Correlation(TotalCharges, tenure x MonthlyCharges): 0.9996


**Decision: fill the 11 blank `TotalCharges` with 0 (instead of deleting rows or using the mean/median).**

**Reason:**
- The blanks are **not random**: they occur exactly for the 11 customers with `tenure = 0`, i.e. customers
  who have just joined and have not been billed yet.
- For everyone else, `TotalCharges` ≈ `tenure × MonthlyCharges` (correlation 0.9996), so the logically
  correct total for a 0-month customer is **0**.
- Mean/median imputation would give brand-new customers a lifetime bill of over a thousand — clearly wrong.
  Dropping rows would work too (only 0.16% of data) but would throw away valid customers for no reason.

In [4]:
df["TotalCharges"] = total_charges_numeric.fillna(0.0)
print("Missing TotalCharges after fix:", df["TotalCharges"].isna().sum())
print("dtype:", df["TotalCharges"].dtype)

Missing TotalCharges after fix: 0
dtype: float64


## 2. `SeniorCitizen`: encoded as 0/1 while similar columns use Yes/No

`SeniorCitizen` is a **category**, not a quantity, but it is stored as an integer while `Partner`,
`Dependents`, etc. use `Yes`/`No`. Converting it keeps all binary flags consistent and ensures it is
treated as categorical during EDA and preprocessing.

In [5]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["SeniorCitizen"].value_counts()

SeniorCitizen
No     5901
Yes    1142
Name: count, dtype: int64

## 3. Inconsistent categorical values

Look for stray whitespace or case differences (e.g. `"yes"` vs `"Yes"`), which would silently
create extra categories.

In [6]:
categorical_cols = [c for c in df.columns if c not in ("customerID", "tenure", "MonthlyCharges", "TotalCharges")]

issues = {}
for col in categorical_cols:
    values = df[col].astype(str)
    if (values != values.str.strip()).any() or values.str.strip().str.lower().nunique() != values.nunique():
        issues[col] = sorted(values.unique())
print("Columns with whitespace/case inconsistencies:", issues or "none")

Columns with whitespace/case inconsistencies: none


**`"No internet service"` and `"No phone service"`** appear in the add-on columns. These are *not*
errors: they are consistent and simply repeat information already in `InternetService` /
`PhoneService`. Let's verify that they line up exactly.

In [7]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
no_internet = df["InternetService"].eq("No")
print("Add-on columns say 'No internet service' exactly when InternetService == 'No':",
      all((df[c] == "No internet service").equals(no_internet) for c in addon_cols))
print("MultipleLines says 'No phone service' exactly when PhoneService == 'No':",
      (df["MultipleLines"] == "No phone service").equals(df["PhoneService"].eq("No")))

Add-on columns say 'No internet service' exactly when InternetService == 'No': True
MultipleLines says 'No phone service' exactly when PhoneService == 'No': True


**Decision: keep "No internet service" / "No phone service" as separate categories.**

**Reason:** they are perfectly consistent with `InternetService` / `PhoneService`, so they are not
data errors. Merging them into "No" would lose nothing, but keeping them is also harmless and is
the more transparent choice. The redundancy is noted (it creates correlated one-hot columns; regularised
models handle this fine).

## 4. Duplicates

Notebook 02 found no fully duplicated rows and no repeated `customerID`. Some customers share an
identical feature profile (all columns except the ID), but they have **different IDs** — these are
genuinely different customers with common plans, so they are kept.

In [8]:
print("Duplicated customerID:", df["customerID"].duplicated().sum())
print("Identical profiles (excluding ID):", df.drop(columns="customerID").duplicated().sum())

Duplicated customerID: 0
Identical profiles (excluding ID): 22


## 5. Invalid or suspicious values

Check the numeric columns for impossible values (negatives, totals smaller than one month's charge)
and for outliers using the IQR rule. Outliers are only **flagged**, never removed automatically.

In [9]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
print("Negative values:", (df[numeric_cols] < 0).sum().to_dict())

# A customer with tenure >= 1 should have been charged at least roughly one month
billed = df[df["tenure"] >= 1]
print("TotalCharges lower than 1 month of MonthlyCharges:",
      (billed["TotalCharges"] < billed["MonthlyCharges"] * 0.5).sum())

def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

print("IQR outliers:", {col: iqr_outlier_count(df[col]) for col in numeric_cols})

Negative values: {'tenure': 0, 'MonthlyCharges': 0, 'TotalCharges': 0}
TotalCharges lower than 1 month of MonthlyCharges: 0
IQR outliers: {'tenure': 0, 'MonthlyCharges': 0, 'TotalCharges': 0}


**Observations**
- No negative values and no customer whose total is implausibly small for their tenure.
- The IQR rule flags **no outliers** in any numeric column — the values are within realistic business ranges,
  so no outlier treatment is needed.

## 6. Unnecessary columns

`customerID` is a unique identifier. It carries no behavioural information, and a model could
only use it to memorise individual rows. It is dropped.

In [10]:
df = df.drop(columns="customerID")
df.shape

(7043, 20)

## 7. Final check and save

The target is kept as `Yes`/`No` here because it reads better in EDA plots; it is converted to
1/0 in notebook 05 just before modelling.

In [11]:
assert df.isna().sum().sum() == 0
assert df["TotalCharges"].dtype == "float64"
print(df.dtypes.value_counts().to_string())

df.to_csv(CLEAN_DATA_PATH, index=False)
print(f"Saved {df.shape} -> {CLEAN_DATA_PATH.relative_to(PROJECT_ROOT)}")

str        17
float64     2
int64       1
Saved (7043, 20) -> data/churn_cleaned.csv


## Conclusion

| Action | Rows affected | Why |
|---|---|---|
| `TotalCharges` text → float; 11 blanks → 0 | 11 | New customers (tenure 0) not yet billed |
| `SeniorCitizen` 0/1 → No/Yes | all | Consistent encoding of binary categories |
| Dropped `customerID` | – | Identifier, no predictive value |
| Duplicates / outliers | 0 removed | None found that were genuine errors |

The clean dataset has **7,043 rows × 20 columns** and no missing values. It is saved to
`data/churn_cleaned.csv`, which is the starting point for all following notebooks.